# Stock Data Exploration

This notebook performs exploratory data analysis (EDA) on monthly market and macroeconomic indicators.

Dataset: `data/combined_monthly.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_PATH = "data/combined_monthly.csv"
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.head()

## 1. Data Snapshot

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nColumn types:")
print(df.dtypes)

print("\nDate range:")
print(df["Date"].min(), "to", df["Date"].max())

df.describe(include="all").transpose()

## 2. Data Quality Checks

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

display(missing.to_frame("missing_values"))
print(f"Duplicate rows: {duplicates}")

## 3. Time-Series Trends

In [ ]:
numeric_cols = [c for c in df.columns if c != "Date"]

fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(14, 3 * len(numeric_cols)), sharex=True)

for ax, col in zip(axes, numeric_cols):
    ax.plot(df["Date"], df[col], linewidth=1.8)
    ax.set_title(col)
    ax.set_ylabel(col)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
corr = df.drop(columns=["Date"]).corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

## 5. SP500 Return Behavior

In [ ]:
df["SP500_Return"] = df["SP500"].pct_change()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["SP500_Return"].dropna(), bins=30, kde=True, ax=ax[0])
ax[0].set_title("Distribution of Monthly SP500 Returns")

sns.boxplot(x=df["SP500_Return"].dropna(), ax=ax[1])
ax[1].set_title("Boxplot of Monthly SP500 Returns")

plt.tight_layout()
plt.show()

## 6. Rolling Trend and Pairwise View

In [ ]:
df["SP500_12M_MA"] = df["SP500"].rolling(window=12).mean()

plt.figure(figsize=(13, 5))
plt.plot(df["Date"], df["SP500"], label="SP500", alpha=0.7)
plt.plot(df["Date"], df["SP500_12M_MA"], label="SP500 12-Month MA", linewidth=2.2)
plt.title("SP500 and 12-Month Moving Average")
plt.xlabel("Date")
plt.ylabel("Index Level")
plt.legend()
plt.tight_layout()
plt.show()

sns.pairplot(df[["SP500", "Crude_Oil", "US_10Y_Yield", "CPI", "Unemployment_Rate"]].dropna(), diag_kind="kde")
plt.show()

## 7. Next Steps

Use this EDA to decide:
- Which features are strongly associated with `SP500`.
- Whether lag features (for example, 1, 3, 6 month lags) may help prediction.
- Whether stationarity transformations (returns/differences) are needed for modeling.